In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

In [2]:
FILE = 'games-players-processed/games_training_set_10.csv'

### Leitura

In [6]:
df = pd.read_csv(FILE)

C:\Users\duilio\AppData\Local\Temp\ipykernel_12900\1324869264.py:1: DtypeWarning: Columns (0: seasonYear_home_p1_home_x, 1: seasonYear_home_p2_home_x, 2: seasonYear_home_p3_home_x, 3: seasonYear_home_p4_home_x, 4: seasonYear_home_p5_home_x, 5: seasonYear_away_p1_home_x, 6: seasonYear_away_p2_home_x, 7: seasonYear_away_p3_home_x, 8: seasonYear_away_p4_home_x, 9: seasonYear_away_p5_home_x, 10: seasonYear_home_p1_away_x, 11: seasonYear_home_p2_away_x, 12: seasonYear_home_p3_away_x, 13: seasonYear_home_p4_away_x, 14: seasonYear_home_p5_away_x, 15: seasonYear_away_p1_away_x, 16: seasonYear_away_p2_away_x, 17: seasonYear_away_p3_away_x, 18: seasonYear_away_p4_away_x, 19: seasonYear_away_p5_away_x, 20: seasonYear_y, 21: refereeName_y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


### Pré-processamento

In [7]:
# Drop non-numeric columns
df = df.select_dtypes(exclude=['category', 'object'])

# Drop columns with more than 80% missing values
threshold = len(df) * 0.8
df = df.dropna(thresh=threshold, axis=1)

# Fill remaining missing values with the mean of each column
df = df.fillna(df.mean())

# Transpose, drop duplicate 'rows' (original columns), and transpose back
df = df.T.drop_duplicates().T 

# Add a new column 'outcome_y' based on the comparison of 'homeScoreNormalTime_y' and 'awayScoreNormalTime_y'
df['outcome_y'] = np.where(df['homeScoreNormalTime_y'] > df['awayScoreNormalTime_y'], 'home', np.where(df['homeScoreNormalTime_y'] < df['awayScoreNormalTime_y'], 'away_draw', 'away_draw'))

# Sort the DataFrame by 'startTimestamp_y' in ascending order
df = df.sort_values(by='startTimestamp_y', ascending=True)

# Drop columns that start with 'gameId', 'homeTeamFoundationDateTimestamp', 'homeTeamId', 'awayTeamFoundationDateTimestamp', or 'awayTeamId'
df = df.filter(regex='^(?!gameId|homeTeamFoundationDateTimestamp|homeTeamId|awayTeamFoundationDateTimestamp|awayTeamId|startTimestamp|round).*$')

In [8]:
df

,tournamentId_home_p1_home_x,hasPlayerStatistics_home_p1_home_x,homeScoreNormalTime_home_p1_home_x,homeScorePeriod1_home_p1_home_x,awayScoreNormalTime_home_p1_home_x,awayScorePeriod1_home_p1_home_x,refereeId_home_p1_home_x,homeBallpossession_home_p1_home_x,awayBallpossession_home_p1_home_x,homeTotalshotsongoal_home_p1_home_x,...,awayGoalkicks_y,homeBigchancecreated_y,awayBigchancecreated_y,homeFouledfinalthird_y,awayFouledfinalthird_y,homeFinalthirdentries_y,awayFinalthirdentries_y,homeWontacklepercent_y,awayWontacklepercent_y,outcome_y
13,83,True,2,0.0,2,1.0,243629.0,43.0,57.0,10.0,...,12.0,6.0,1.0,2.0,4.0,46.0,57.0,20.0,10.0,home
14,83,True,2,0.0,0,0.0,243133.0,70.0,30.0,21.0,...,16.0,2.152073,1.501161,2.942351,2.388853,56.967174,51.00777,10.324397,10.16323,away_draw
15,83,True,2,1.0,0,0.0,554152.0,42.0,58.0,7.0,...,10.0,2.152073,1.501161,2.942351,2.388853,56.967174,51.00777,10.324397,10.16323,away_draw
17,83,True,0,0.0,3,1.0,243153.0,49.0,51.0,14.0,...,5.0,2.152073,1.501161,2.942351,2.388853,56.967174,51.00777,10.324397,10.16323,away_draw
16,83,True,3,2.0,0,0.0,243135.0,62.0,38.0,17.0,...,7.0,2.152073,1.501161,2.942351,2.388853,56.967174,51.00777,10.324397,10.16323,home
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5126,83,True,2,2.0,1,0.0,788854.0,39.0,61.0,7.0,...,13.0,3.0,0.0,4.0,1.0,70.0,41.0,3.0,4.0,home
5127,2929,True,1,0.0,1,1.0,243495.0,68.0,32.0,16.0,...,8.0,2.0,2.0,1.0,3.0,72.0,37.0,4.0,5.0,home
6866,91443,True,1,0.0,1,1.0,786086.0,64.0,36.0,10.0,...,5.0,0.0,5.0,0.0,2.0,42.0,60.0,10.0,16.0,away_draw
6867,91443,True,2,2.0,0,0.0,787846.0,43.0,57.0,20.0,...,3.0,1.0,4.0,5.0,1.0,35.0,48.0,6.0,13.0,away_draw


In [9]:
df['outcome_y'].value_counts()

outcome_y
away_draw    3622
home         3246
Name: count, dtype: int64

In [10]:
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

X_train = train_data.filter(regex='_x$')
X_test = test_data.filter(regex='_x$')

y_train = train_data['outcome_y']
y_test = test_data['outcome_y']

#scaler = StandardScaler().set_output(transform="pandas")
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.transform(X_test)

In [14]:
model = RandomForestClassifier(n_estimators=2000, min_samples_split=7, max_features='sqrt', random_state=42, verbose=1, n_jobs=8)
model.fit(X_train, y_train)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    2.8s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    7.1s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:   12.8s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:   20.3s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:   32.6s
[Parallel(n_jobs=8)]: Done 2000 out of 2000 | elapsed:   37.2s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",2000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",7
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [47]:
def predict_with_ambiguity(model, X, threshold=0.005, ambiguous_class=1):
    probs = model.predict_proba(X)
    sorted_indices = np.argsort(probs, axis=1)
    top2_indices = sorted_indices[:, -2:]
    top_probs = np.take_along_axis(probs, top2_indices, axis=1)
    diff = top_probs[:, 1] - top_probs[:, 0]
    predictions = np.argmax(probs, axis=1)
    predictions[diff <= threshold] = ambiguous_class
    return model.classes_[predictions]

In [15]:
# Predict on test data
#y_pred = predict_with_ambiguity(model, X_test)
y_pred = model.predict(X_test)
print(pd.Series(y_pred).value_counts())

print(y_test)
print(y_pred)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred, digits=4))

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:    0.4s


away_draw    881
home         493
Name: count, dtype: int64
4919         home
4918    away_draw
4920         home
6708         home
2860         home
          ...    
5126         home
5127         home
6866    away_draw
6867    away_draw
5128    away_draw
Name: outcome_y, Length: 1374, dtype: str
['home' 'home' 'home' ... 'away_draw' 'away_draw' 'home']
Accuracy: 0.5858806404657934
              precision    recall  f1-score   support

   away_draw     0.5925    0.7131    0.6472       732
        home     0.5740    0.4408    0.4987       642

    accuracy                         0.5859      1374
   macro avg     0.5833    0.5770    0.5730      1374
weighted avg     0.5839    0.5859    0.5778      1374



[Parallel(n_jobs=8)]: Done 2000 out of 2000 | elapsed:    0.4s finished


In [29]:
# Assume 'model' is your trained model and 'X' is your training data
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort and print
ordered_imp = feature_imp_df.sort_values(by='Importance', ascending=False)
print(ordered_imp[:10])

                                   Feature  Importance
109      homeAccuratepasses_home_p2_home_x    0.002111
68    awayFinalthirdentries_home_p1_home_x    0.002091
1198             awayPasses_away_p3_away_x    0.001996
503              homePasses_away_p3_home_x    0.001914
525      homeAccuratepasses_away_p3_home_x    0.001914
1220     awayAccuratepasses_away_p3_away_x    0.001845
87               homePasses_home_p2_home_x    0.001798
1219     homeAccuratepasses_away_p3_away_x    0.001723
1151     awayAccuratepasses_away_p2_away_x    0.001696
345   awayFinalthirdentries_home_p5_home_x    0.001659


In [21]:
probs = model.predict_proba(X_test)

sorted_probs = sorted(probs, key=lambda x: abs(x[0] - x[1]), reverse=True)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 2000 out of 2000 | elapsed:    0.4s finished


In [23]:
ctotal = 0
ccorrect = 0
ccorrectaway = 0
ccorrecthome = 0
for i, prob in enumerate(probs):
    if abs(prob[1] - prob[0]) >= 0.20:
        ctotal += 1
        if y_test.iloc[i] == model.classes_[np.argmax(prob)]:
            ccorrect += 1
            ccorrectaway += 1 if y_test.iloc[i] == 'away_draw' else 0
            ccorrecthome += 1 if y_test.iloc[i] == 'home' else 0

print(f"Confident predictions (as a percentage of all predictions): {ctotal/len(probs)*100:.2f}%")
print(f"Correct confident predictions: {ccorrect}")
print(f"Total confident predictions: {ctotal}")
print(f"Confidence precision: {ccorrect/ctotal*100:.2f}%")
print(f"Correct away_draw predictions: {ccorrectaway}")
print(f"Correct home predictions: {ccorrecthome}")

Confident predictions (as a percentage of all predictions): 9.75%
Correct confident predictions: 97
Total confident predictions: 134
Confidence precision: 72.39%
Correct away_draw predictions: 87
Correct home predictions: 10


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Initialize the classifier
rfc = RandomForestClassifier(random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [400, 500, 600],
    'max_depth': [None, 30, 40],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [5, 7, 9],
}

# Setup GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=rfc, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit to training data
grid_search.fit(X_train, y_train)

# Access the best parameters and model
print(f"Best Parameters: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_split': 9, 'n_estimators': 500}


In [19]:
# 1. Convert the cv_results_ dictionary into a pandas DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# 2. Select the most useful columns to keep things readable
# (This includes the rank, the final mean test score, and your hyperparameter choices)
columns_to_keep = [
    'rank_test_score', 
    'mean_test_score', 
    'std_test_score', 
    'param_n_estimators', 
    'param_max_depth', 
    'param_max_features', 
    'param_min_samples_split'
]
results_df = results_df[columns_to_keep]

# 3. Sort the DataFrame so the best models (rank 1) appear at the top
results_df_sorted = results_df.sort_values(by='rank_test_score')

# 4. Display the top 10 configurations
results_df_sorted.head(10)

,rank_test_score,mean_test_score,std_test_score,param_n_estimators,param_max_depth,param_max_features,param_min_samples_split
25,1,0.582457,0.019463,500,30,sqrt,9
50,2,0.582092,0.019515,600,40,log2,7
14,3,0.581910,0.019363,600,None,log2,7
43,4,0.581728,0.020537,500,40,sqrt,9
31,5,0.581364,0.019446,500,30,log2,7
7,6,0.581182,0.019875,500,None,sqrt,9
23,7,0.580998,0.019847,600,30,sqrt,7
26,8,0.580272,0.016460,600,30,sqrt,9
44,9,0.580271,0.015974,600,40,sqrt,9
24,10,0.580090,0.014597,400,30,sqrt,9
